# Grounded Carbon-Market QA: **v2 research-based run** (Karan)
**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Allow Google Drive access when asked.

This is a separate run from the team's original notebook. It changes the method, based on a literature review
(`reports/Grounded RAG finetuning methods.md`):

| Change | Why (paper) |
|---|---|
| **Quote-then-answer** targets: `Quote: [S#] "…"` then `Answer: … [S#]` | RAFT (arXiv 2403.10131), FRONT (2408.04568) |
| **Loss on the answer only** (the original trained on the 2k-token prompt too) | standard completion-only SFT |
| Refusals capped at **~28%** of SFT data | ChatQA (2401.10225): too many refusals hurts |
| **SFT → DPO** on ~800 auto-built pairs (answer>refusal, refusal>made-up answer, right>wrong citation, faithful>contradiction) | Trust-Align (2409.11242), symmetric refusal-DPO (2603.19251), Ground-GRPO (2506.15522) |
| Judge from a **different family** (Gemma-3-12B) + **HHEM** checker + exact citation/quote checks | self-preference bias (Panickssery et al. 2024), HHEM-2.1-open |
| Grounded-refusal F1, McNemar test, paired bootstrap CIs | Trust-Score (ICLR 2025), Miller 2024 |

**Lexsi toolkits:** CuratorKIT-style data pipeline · **AlignTune** `create_sft_trainer` + `create_rl_trainer(algorithm="dpo")` · **AuditKIT** metrics + `ak.compare`.

**Resumable:** everything is saved to Google Drive (`MyDrive/lexsi_karan_v2`). If Colab disconnects, just
**Run all** again: finished stages are skipped. Total ≈ 4–5 h of GPU time, may need two sessions.

## 1. Setup: Drive, code, settings

In [ ]:
import os, shutil, json, time, subprocess
from google.colab import drive
drive.mount('/content/drive')
SAVE = '/content/drive/MyDrive/lexsi_karan_v2'
os.makedirs(f'{SAVE}/data/work', exist_ok=True); os.makedirs(f'{SAVE}/outputs', exist_ok=True)
if not os.path.exists('/content/proj'):
    !git clone -q https://github.com/snehapal0905/grounded-carbon-llm /content/proj
os.chdir('/content/proj')
for f in ['chunks.jsonl', 'llm_cache.jsonl', 'ingest_stats.json']:   # seed Drive once (Groq answers are reused)
    if not os.path.exists(f'{SAVE}/data/work/{f}'):
        shutil.copy(f'data/work/{f}', f'{SAVE}/data/work/{f}')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception: print('HF_TOKEN not set (only needed to publish to Hugging Face)')
GEN = 'ollama_chat/qwen2.5:14b'      # writes data (same as the team run)
JUDGE = 'ollama_chat/gemma3:12b'     # grades answers: NOT Qwen, so no self-preference bias
os.environ.update({
    'DATA_DIR': f'{SAVE}/data', 'OUT_DIR': f'{SAVE}/outputs', 'MERGED_DIR': '/content/merged_sft',
    'LITELLM_LOCAL_MODEL_COST_MAP': 'True',
    'GEN_MODEL': GEN, 'GATE_MODEL': GEN, 'VERIFY_MODEL': GEN,
    'JUDGE_MODEL': JUDGE, 'AUDIT_JUDGE': 'litellm:' + JUDGE,
    'CACHE_FALLBACK_MODELS': 'groq/openai/gpt-oss-120b',
    'LLM_CONCURRENCY': '4', 'LLM_TIMEOUT': '600', 'OLLAMA_NUM_PARALLEL': '4',
    'N_PREF': '800', 'NEG_RATE': '0.25', 'MAX_REFUSAL_SHARE': '0.28',
})
W, O = f'{SAVE}/data/work', f'{SAVE}/outputs'
print('Drive folder:', SAVE); print(sorted(os.listdir(W)))

## 2. Write the v2 code (research-based changes) over the team repo
Run these cells as they are. They overwrite 8 files with the v2 versions (branch `karan-v2`).

In [ ]:
%%writefile common/gpu.py
"""What GPU are we on? Picks dtype / 4-bit / batch sizes so the same code runs on a T4 and on RunPod."""
import torch


def native_bf16():
    if not torch.cuda.is_available():
        return False
    try:
        return torch.cuda.is_bf16_supported(including_emulation=False)  # T4 only *emulates* bf16
    except TypeError:
        return torch.cuda.get_device_capability(0)[0] >= 8


def gpu_gib():
    return torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0


BF16 = native_bf16()                 # Ampere+ (A100, L40S, 4090, H100)
BIG = gpu_gib() >= 30                 # 40-80 GB cards: no 4-bit, bigger batches
DTYPE = torch.bfloat16 if BF16 else torch.float16
DTYPE_NAME = "bfloat16" if BF16 else "float16"
GEN_BATCH = 32 if BIG else 8


def describe():
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    return f"{name}, {gpu_gib():.0f} GiB, dtype={DTYPE_NAME}, 4-bit={'no' if BIG else 'yes'}"

In [ ]:
%%writefile common/prompts.py
"""Prompt formats shared by data building, training, evaluation and the demo (v2: quote-then-answer).

v2 change (RAFT, Zhang et al. 2024; FRONT, Huang et al. 2024): when the answer is in the sources the
model first copies the exact supporting sentence, then answers with a citation:

    Quote: [S2] "<sentence copied verbatim from source 2>"
    Answer: <1-3 sentence answer> [S2]

When it is not in the sources it replies with exactly REFUSAL. The same system prompt is used for
training, for the strict baseline, and for the fine-tuned model, so gains come from training only.
"""
import re

REFUSAL = "Not found in the provided documents."

SYSTEM_STRICT = (
    "You are a compliance assistant for carbon-market rules. Answer ONLY from the "
    "numbered sources in the user's message.\n"
    "Rules:\n"
    "1. Every statement must be supported by the sources. Never use outside knowledge.\n"
    "2. Cite the sources you used with their tags, e.g. [S2] or [S1][S3].\n"
    f'3. If the sources do not contain the answer, reply exactly: "{REFUSAL}" Do not guess.\n'
    "4. Be concise: 1-4 sentences."
)

SYSTEM_QUOTE = (
    "You are a compliance assistant for carbon-market rules. Answer ONLY from the "
    "numbered sources in the user's message. Never use outside knowledge and never guess.\n"
    "If the sources contain the answer, reply in exactly this format:\n"
    'Quote: [S#] "<the supporting sentence, copied word for word from that source>"\n'
    "Answer: <1-3 sentence answer> [S#]\n"
    f"If the sources do not contain the answer, reply with exactly this sentence and nothing else: {REFUSAL}"
)

# Weak prompt for the "plain RAG" baseline.
SYSTEM_SIMPLE = "You are a helpful assistant. Use the context to answer the question."


def format_sources(chunks):
    return "\n\n".join(
        f"[S{i + 1}] ({c['doc']}, p.{c['page']})\n{c['text']}" for i, c in enumerate(chunks)
    )


def user_message(question, chunks):
    return f"Sources:\n{format_sources(chunks)}\n\nQuestion: {question}"


def build_messages(question, chunks, system=SYSTEM_STRICT, answer=None):
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": user_message(question, chunks)}]
    if answer is not None:
        msgs.append({"role": "assistant", "content": answer})
    return msgs


_REFUSAL_PATTERNS = re.compile(
    r"not found in the provided documents|(?:sources?|context|documents?|passages?) (?:do|does) not "
    r"(?:contain|provide|mention|specify|include|say)|no information (?:about|on|regarding)|"
    r"not (?:mentioned|specified|covered|stated) in the (?:sources?|context|documents?)|"
    r"cannot be (?:determined|answered) from",
    re.I,
)


def looks_like_refusal(text: str) -> bool:
    """Cheap heuristic (the LLM judge makes the final call during evaluation)."""
    return bool(_REFUSAL_PATTERNS.search(text or ""))


def cited_ids(text: str):
    return sorted({int(m) for m in re.findall(r"\[S(\d+)\]", text or "")})


# ---------------- v2 helpers: quote-then-answer ----------------
_SENT = re.compile(r"(?<=[.;:!?])\s+|\n+")
_WORD = re.compile(r"[a-z0-9]+(?:\.[0-9]+)*")
_STOP = {"the", "a", "an", "of", "to", "and", "or", "in", "is", "are", "be", "for", "on", "by", "with",
         "that", "this", "as", "at", "it", "from", "must", "shall", "s", "which", "under", "any", "its"}


def _content(t):
    return [w for w in _WORD.findall((t or "").lower()) if w not in _STOP]


def best_sentence(answer, text, max_words=60):
    """Sentence of `text` (verbatim) that best supports `answer` (content-word overlap)."""
    ans = set(_content(answer))
    best, best_s = "", -1.0
    for s in (x.strip() for x in _SENT.split(text or "")):
        if len(s.split()) < 4:
            continue
        toks = _content(s)
        if not toks:
            continue
        score = len(ans & set(toks)) + 0.01 * min(len(toks), 30)
        if score > best_s:
            best, best_s = s, score
    words = best.split()
    if len(words) > max_words:  # keep a verbatim window around the densest overlap
        hits = [i for i, w in enumerate(words) if set(_content(w)) & ans]
        mid = hits[len(hits) // 2] if hits else 0
        start = max(0, min(mid - max_words // 2, len(words) - max_words))
        best = " ".join(words[start:start + max_words])
    return best


def quote_answer(answer, quote, sid):
    answer = re.sub(r"\s*\[S\d+\]\s*$", "", answer.strip())
    return f'Quote: [S{sid}] "{quote}"\nAnswer: {answer} [S{sid}]'


def final_answer(text):
    """The user-facing answer: the part after 'Answer:' if the model used the quote format."""
    t = (text or "").strip()
    m = re.search(r"(?im)^\s*answer\s*:\s*(.*)\Z", t, re.S)
    return m.group(1).strip() if m else t


def quotes(text):
    """[(source_id, quoted_text)] from 'Quote: [S#] "..."' lines."""
    return [(int(a), b) for a, b in re.findall(r'\[S(\d+)\]\s*["“](.+?)["”]', text or "", re.S)
            if re.search(r"(?i)quote", text or "")]


def is_refusal(text):
    a = final_answer(text)
    return a.startswith(REFUSAL) or (len(a) < 200 and looks_like_refusal(a))


def _norm(t):
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", (t or "").lower())).strip()


def quote_is_verbatim(quote, source_text):
    q = _norm(quote)
    return bool(q) and q in _norm(source_text)

In [ ]:
%%writefile common/model.py
"""Load a base model (original or merged-SFT), optionally attach a LoRA adapter, toggle it on/off.

v2: `base` can be the merged SFT checkpoint (for the DPO adapter), and max_new_tokens is larger
because quote-then-answer outputs are longer than plain answers.
"""
import contextlib
from pathlib import Path

import torch

from common.gpu import DTYPE, GEN_BATCH
from config import BASE_MODEL


def load_causal_lm(path, **kw):
    from transformers import AutoModelForCausalLM
    try:
        return AutoModelForCausalLM.from_pretrained(path, dtype=DTYPE, **kw)
    except TypeError:
        return AutoModelForCausalLM.from_pretrained(path, torch_dtype=DTYPE, **kw)


class Generator:
    def __init__(self, base=BASE_MODEL, adapter_dir=None, load_in_4bit=False):
        from transformers import AutoTokenizer

        self.tok = AutoTokenizer.from_pretrained(str(base))
        self.tok.padding_side = "left"
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        kw = {"device_map": {"": 0}}
        if load_in_4bit:
            from transformers import BitsAndBytesConfig
            kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True,
                                                           bnb_4bit_compute_dtype=DTYPE)
        model = load_causal_lm(str(base), **kw)
        self.has_adapter = adapter_dir is not None and (Path(adapter_dir) / "adapter_config.json").exists()
        if self.has_adapter:
            from peft import PeftModel
            model = PeftModel.from_pretrained(model, str(adapter_dir))
        elif adapter_dir is not None:
            print(f"! No adapter at {adapter_dir}")
        self.model = model.eval()

    def _ctx(self, tuned):
        if tuned:
            if not self.has_adapter:
                raise RuntimeError("Fine-tuned system requested but no adapter loaded.")
            return contextlib.nullcontext()
        return self.model.disable_adapter() if self.has_adapter else contextlib.nullcontext()

    @torch.inference_mode()
    def generate(self, batch_messages, tuned=False, max_new_tokens=320, batch_size=GEN_BATCH):
        from tqdm import tqdm
        outs = []
        # sort by length so batches pad less; restore order after
        order = sorted(range(len(batch_messages)), key=lambda i: len(str(batch_messages[i])))
        with self._ctx(tuned):
            for s in tqdm(range(0, len(order), batch_size), desc="generate"):
                idx = order[s:s + batch_size]
                prompts = [self.tok.apply_chat_template(batch_messages[i], tokenize=False,
                                                        add_generation_prompt=True) for i in idx]
                enc = self.tok(prompts, return_tensors="pt", padding=True).to(self.model.device)
                gen = self.model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                          temperature=None, top_p=None, top_k=None,
                                          pad_token_id=self.tok.pad_token_id)
                dec = self.tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
                outs += list(zip(idx, dec))
        outs.sort()
        return [o.strip() for _, o in outs]

    def free(self):
        import gc
        del self.model
        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
%%writefile scripts/04_build_sets.py
"""Step 4 (v2) - SFT set (quote-then-answer, completion-only), DPO preference pairs, test candidates.

Research basis (see reports/Grounded RAG finetuning methods.md):
  * RAFT (arXiv 2403.10131) / FRONT (2408.04568): gold passage among distractors, target first quotes
    the supporting sentence verbatim, then answers with a citation.
  * ChatQA (2401.10225): refusals help up to a point, then hurt -> refusal share capped at ~28%.
  * Trust-Align (2409.11242) + symmetric refusal-DPO (2603.19251): preference pairs where the SAME
    question appears with and without its evidence; every rejected answer carries a citation.

SFT rows are {"prompt": [system, user], "completion": [assistant]} so the loss covers only the answer
(the original repo trained on the ~2k-token prompt too).

Pair mix (N_PREF total, default 800):
  25.0%  answer_over_refusal       gold present: quote+answer  >  refusal          (fights over-refusal)
  50.0%  refusal_over_answer       evidence removed / unanswerable: refusal  >  cited made-up answer
  12.5%  right_over_wrong_citation gold present: correct [S#] + quote  >  wrong [S#] + other quote
  12.5%  faithful_over_contradiction gold present: correct  >  same answer with one fact changed

Outputs: outputs/sft/{train,validation}.jsonl, outputs/pref/train.jsonl, outputs/test_candidates.csv
"""
import csv
import json
import os
import random
import re
import sys
from collections import Counter
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from common.llm import chat, pmap  # noqa: E402
from common.prompts import (REFUSAL, SYSTEM_QUOTE, best_sentence, build_messages,  # noqa: E402
                            quote_answer)
from common.retrieval import Index, answer_recall, is_neighbor, load_chunks  # noqa: E402
from config import GEN_MODEL, OUT, SEED, TEST_PER_CLASS, TOP_K, WORK  # noqa: E402

NEG_RATE = float(os.getenv("NEG_RATE", "0.25"))          # share of positives also used as hard negatives
MAX_REFUSAL_SHARE = float(os.getenv("MAX_REFUSAL_SHARE", "0.28"))
N_PREF = int(os.getenv("N_PREF", "800"))

BAD_Q = re.compile(r"\b(the|this|above|given|provided) (passage|section|text|excerpt|paragraph)\b|"
                   r"according to the (passage|text)", re.I)

MADEUP_PROMPT = """A carbon-market analyst asks: "{q}"
Write the kind of confident, specific answer an overconfident assistant would give from memory:
1-2 sentences, include a concrete number, date, fee or rule. Output only the answer."""

CONTRADICT_PROMPT = """Rewrite this answer so it sounds equally confident but changes exactly ONE key fact
(a number, duration, percentage, named body, or requirement) to a wrong one. Output only the rewritten answer.

Answer: {a}"""


def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f]


def row(question, ctx, target):
    m = build_messages(question, ctx, system=SYSTEM_QUOTE)
    return {"prompt": m, "completion": [{"role": "assistant", "content": target}]}


def pair(question, ctx, chosen, rejected, kind):
    return {"prompt": build_messages(question, ctx, system=SYSTEM_QUOTE),
            "chosen": [{"role": "assistant", "content": chosen}],
            "rejected": [{"role": "assistant", "content": rejected}], "kind": kind}


def main():
    rng = random.Random(SEED)
    chunks = load_chunks()
    by_id = {c["chunk_id"]: c for c in chunks}
    index = Index(chunks)
    train_mask = index.mask(splits={"train"})

    qa = read_jsonl(WORK / "qa_raw.jsonl")
    unans = read_jsonl(WORK / "unanswerable.jsonl")
    n0 = len(qa)
    qa = [r for r in qa if not BAD_Q.search(r["question"]) and len(r["answer"]) > 2]
    print(f"QA pairs: {n0} -> {len(qa)} after dropping non-self-contained questions")

    pos, negs, unas = [], [], []
    for r in qa:
        if r["split"] != "train":
            continue
        gold = by_id[r["chunk_id"]]
        cands = index.search(r["question"], k=TOP_K + 6, mask=train_mask)
        distract = [c for c in cands if c["chunk_id"] != gold["chunk_id"] and not is_neighbor(c, gold)]
        if len(distract) < TOP_K - 1:
            continue
        ctx = distract[:TOP_K - 1] + [gold]
        rng.shuffle(ctx)
        sid = ctx.index(gold) + 1
        quote = best_sentence(r["answer"], gold["text"])
        if not quote:
            continue
        pos.append({"q": r["question"], "a": r["answer"], "ctx": ctx, "sid": sid, "quote": quote,
                    "target": quote_answer(r["answer"], quote, sid)})
        if rng.random() < NEG_RATE:
            neg = [c for c in cands if c["chunk_id"] != gold["chunk_id"] and not is_neighbor(c, gold)
                   and answer_recall(r["answer"], c["text"]) < 0.5][:TOP_K]
            if len(neg) == TOP_K:
                negs.append({"q": r["question"], "a": r["answer"], "ctx": neg})
    for q in unans:
        if q["split"] == "train":
            unas.append({"q": q["question"], "kind": q["kind"],
                         "ctx": index.search(q["question"], k=TOP_K, mask=train_mask)})

    # ---- SFT set, refusal share capped (ChatQA: too many refusals hurts) ----
    refusals = [("hard_negative", n) for n in negs] + [("unanswerable", u) for u in unas]
    rng.shuffle(refusals)
    max_ref = int(MAX_REFUSAL_SHARE * len(pos) / (1 - MAX_REFUSAL_SHARE))
    refusals = refusals[:max_ref]
    sft = [("positive", row(p["q"], p["ctx"], p["target"])) for p in pos] + \
          [(k, row(x["q"], x["ctx"], REFUSAL)) for k, x in refusals]
    rng.shuffle(sft)
    n_val = max(int(len(sft) * 0.04), 16)
    d = OUT / "sft"
    d.mkdir(parents=True, exist_ok=True)
    for name, part in (("validation", sft[:n_val]), ("train", sft[n_val:])):
        with open(d / f"{name}.jsonl", "w", encoding="utf-8") as f:
            for _, e in part:
                f.write(json.dumps(e, ensure_ascii=False) + "\n")
    sft_stats = Counter(k for k, _ in sft)
    print(f"SFT: {len(sft) - n_val} train / {n_val} val {dict(sft_stats)} "
          f"refusal share {1 - sft_stats['positive'] / len(sft):.0%}")

    # ---- DPO preference pairs ----
    n_ans, n_ref = int(N_PREF * 0.25), int(N_PREF * 0.5)
    n_cit, n_con = int(N_PREF * 0.125), int(N_PREF * 0.125)
    pairs = []
    P = pos[:]
    rng.shuffle(P)
    for p in P[:n_ans]:
        pairs.append(pair(p["q"], p["ctx"], p["target"], REFUSAL, "answer_over_refusal"))
    for p in P[n_ans:n_ans + n_cit]:
        wrong = rng.choice([i for i in range(1, TOP_K + 1) if i != p["sid"]])
        wq = best_sentence(p["a"], p["ctx"][wrong - 1]["text"])
        if wq:
            pairs.append(pair(p["q"], p["ctx"], p["target"], quote_answer(p["a"], wq, wrong),
                              "right_over_wrong_citation"))
    con_src = P[n_ans + n_cit:n_ans + n_cit + n_con]
    bad = pmap(lambda p: chat(GEN_MODEL, [{"role": "user", "content": CONTRADICT_PROMPT.format(a=p["a"])}],
                              temperature=0.7, max_tokens=300), con_src, desc="contradictions")
    for p, b in zip(con_src, bad):
        b = (b or "").strip()
        if b and b.lower() != p["a"].lower():
            pairs.append(pair(p["q"], p["ctx"], p["target"], quote_answer(b, p["quote"], p["sid"]),
                              "faithful_over_contradiction"))
    # refusal > cited answer: hard negatives (the real answer, but its evidence is gone) + unanswerables
    ref_src = [("hn", n) for n in negs] + [("un", u) for u in unas]
    rng.shuffle(ref_src)
    ref_src = ref_src[:n_ref]
    need = [x for k, x in ref_src if k == "un"]
    made = pmap(lambda u: chat(GEN_MODEL, [{"role": "user", "content": MADEUP_PROMPT.format(q=u["q"])}],
                               temperature=0.7, max_tokens=200), need, desc="made-up answers")
    made_by_q = {u["q"]: (m or "").strip() for u, m in zip(need, made)}
    for k, x in ref_src:
        ans = x["a"] if k == "hn" else made_by_q.get(x["q"], "")
        if not ans:
            continue
        sid = rng.randint(1, TOP_K)
        q = best_sentence(ans, x["ctx"][sid - 1]["text"])
        if q:
            pairs.append(pair(x["q"], x["ctx"], REFUSAL, quote_answer(ans, q, sid), "refusal_over_answer"))
    rng.shuffle(pairs)
    pd = OUT / "pref"
    pd.mkdir(parents=True, exist_ok=True)
    with open(pd / "train.jsonl", "w", encoding="utf-8") as f:
        for p in pairs:
            f.write(json.dumps({k: p[k] for k in ("prompt", "chosen", "rejected")}, ensure_ascii=False) + "\n")
    pref_stats = Counter(p["kind"] for p in pairs)
    print(f"DPO pairs: {len(pairs)} {dict(pref_stats)}")

    # ---- test candidates (same construction as the original repo) ----
    test_qa = [r for r in qa if r["split"] == "test"]
    rng.shuffle(test_qa)
    per_chunk, picked = Counter(), []
    for r in test_qa:
        if per_chunk[r["chunk_id"]] < 2:
            per_chunk[r["chunk_id"]] += 1
            picked.append(r)
    picked = picked[:TEST_PER_CLASS]
    rows = [{"id": f"A{i:03d}", "type": "answerable", "kind": "", "question": r["question"],
             "gold_answer": r["answer"], "gold_chunk_id": r["chunk_id"],
             "source_text": by_id[r["chunk_id"]]["text"], "keep": 1} for i, r in enumerate(picked)]
    rows += [{"id": f"U{i:03d}", "type": "unanswerable", "kind": q["kind"], "question": q["question"],
              "gold_answer": REFUSAL, "gold_chunk_id": "", "source_text": "", "keep": 1}
             for i, q in enumerate([q for q in unans if q["split"] == "test"][:TEST_PER_CLASS])]
    with open(OUT / "test_candidates.csv", "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    print(f"Test candidates: {sum(r['type'] == 'answerable' for r in rows)} answerable, "
          f"{sum(r['type'] == 'unanswerable' for r in rows)} unanswerable")
    json.dump({"sft": dict(sft_stats), "sft_train": len(sft) - n_val, "pref": dict(pref_stats),
               "test_rows": len(rows)}, open(WORK / "sft_stats_v2.json", "w"), indent=2)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/05_train.py
"""Step 5 (v2) - SFT (answer-only loss) -> merge -> DPO, all through AlignTune with a plain-TRL fallback.

    python scripts/05_train.py --stage sft   [--smoke]
    python scripts/05_train.py --stage merge
    python scripts/05_train.py --stage dpo   [--smoke]

Settings follow the research report (Trust-Align 2409.11242, Ground-GRPO 2506.15522, Unsloth/TRL DPO docs):
  SFT : LoRA r16/a32 on all 7 projections, 4-bit, lr 2e-4, 2 epochs, loss on the answer only.
  DPO : starts from the MERGED SFT model (AlignTune cannot load an adapter), LoRA, 4-bit,
        beta 0.5, lr 1e-5, 1 epoch, + small NLL term on the chosen answer (rpo_alpha) so DPO does not
        drift into over-refusal; max_prompt_length raised so the sources are never truncated.
Outputs: outputs/adapter_sft/, $MERGED_DIR (local disk), outputs/adapter_dpo/
"""
import argparse
import dataclasses
import json
import os
import shutil
import sys
import traceback
from pathlib import Path

# Must be set before torch is imported: stops the "3.5 GiB reserved but unallocated" fragmentation OOMs.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from common.gpu import BF16, BIG, DTYPE, DTYPE_NAME, describe  # noqa: E402
from config import BASE_MODEL, OUT  # noqa: E402

LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
MERGED_DIR = Path(os.getenv("MERGED_DIR", "/content/merged_sft"))
MAX_LEN, MAX_PROMPT, MAX_COMPLETION = 3072, 2600, 448
# Pairs longer than this are skipped. TRL >= 1.18 scores only the answer tokens (fused LM head), so on a
# 16 GB T4 full-length pairs fit; on older TRL set DPO_MAX_TOKENS=2048.
DPO_MAX_TOKENS = int(os.getenv("DPO_MAX_TOKENS", str(MAX_LEN)))
SAVE_EVERY = 10  # optimizer steps between checkpoints (resume after a disconnect)
# Precision: bf16 on Ampere+ (RunPod), fp16 on T4 (it only emulates bf16 -> very slow).
# 4-bit only on small cards; on 40-80 GB cards plain bf16 LoRA is faster and a bit more accurate.
QUANT = {} if BIG else {"load_in_4bit": True, "bnb_4bit_quant_type": "nf4", "bnb_4bit_compute_dtype": DTYPE_NAME}
PREC = dict(precision="bf16" if BF16 else "fp16", fp16=not BF16, bf16=BF16)
SFT_BS, DPO_BS = (4, 2) if BIG else (1, 1)   # effective batch stays 16


def mk(cls, **kw):
    """Build a TRL config, silently dropping fields this TRL version doesn't have."""
    names = {f.name for f in dataclasses.fields(cls)}
    dropped = [k for k in kw if k not in names]
    if dropped:
        print(f"  ({cls.__name__}: ignoring unsupported {dropped})")
    return cls(**{k: v for k, v in kw.items() if k in names})


def load_train_model(path):
    from transformers import AutoTokenizer, BitsAndBytesConfig
    from common.model import load_causal_lm
    tok = AutoTokenizer.from_pretrained(str(path))
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    if BIG:
        model = load_causal_lm(str(path), device_map={"": 0})
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.enable_input_require_grads()
        return model, tok
    from peft import prepare_model_for_kbit_training
    model = load_causal_lm(str(path), device_map={"": 0}, quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE))
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    return model, tok


def lora():
    from peft import LoraConfig
    return LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=LORA_TARGETS, task_type="CAUSAL_LM")


def fp32_trainable():
    """fp16 mixed precision needs fp32 trainable weights. Some loaders leave LoRA in bf16, which crashes the
    grad scaler ('_amp_foreach_non_finite_check_and_unscale_cuda not implemented for BFloat16')."""
    import torch
    from transformers import Trainer
    orig = Trainer.__init__

    def init(self, *args, **kw):
        orig(self, *args, **kw)
        n = 0
        for p in self.model.parameters():
            if p.requires_grad and p.dtype != torch.float32:
                p.data = p.data.float()
                n += 1
        if n:
            print(f">> cast {n} trainable tensors to fp32 for fp16 training")

    Trainer.__init__ = init


def enable_resume(run_dir):
    """Make every HF Trainer (AlignTune's included) resume from the newest checkpoint in run_dir."""
    from transformers import Trainer
    orig = Trainer.train

    def train(self, resume_from_checkpoint=None, *args, **kw):
        cks = sorted(Path(run_dir).glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
        if resume_from_checkpoint is None and cks:
            resume_from_checkpoint = str(cks[-1])
            print(f">> resuming from {resume_from_checkpoint}")
        return orig(self, resume_from_checkpoint, *args, **kw)

    Trainer.train = train


def short_pairs(data, model_path, dest, limit=None):
    """Write the preference pairs that fit in DPO_MAX_TOKENS (prompt + longer answer) to dest."""
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(str(model_path))
    rows = [json.loads(l) for l in open(data, encoding="utf-8") if l.strip()]
    keep = []
    for r in rows:
        n = max(len(tok(tok.apply_chat_template(r["prompt"] + r[k], tokenize=False),
                        add_special_tokens=False)["input_ids"]) for k in ("chosen", "rejected"))
        if n <= DPO_MAX_TOKENS:
            keep.append(r)
    if limit:
        keep = keep[:limit]
    Path(dest).parent.mkdir(parents=True, exist_ok=True)
    with open(dest, "w", encoding="utf-8") as f:
        for r in keep:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"DPO pairs: kept {len(keep)} of {len(rows)} (<= {DPO_MAX_TOKENS} tokens) -> {dest}")
    return dest


def collect_adapter(run_dir, dest):
    cfgs = [p for p in Path(run_dir).rglob("adapter_config.json")]
    if not cfgs:
        raise FileNotFoundError(f"No adapter_config.json under {run_dir}")
    final = [p for p in cfgs if "checkpoint-" not in str(p)]
    src = (final or sorted(cfgs, key=lambda p: p.stat().st_mtime))[-1].parent
    if Path(dest).exists():
        shutil.rmtree(dest)
    shutil.copytree(src, dest, ignore=shutil.ignore_patterns("optimizer.pt", "*.pth", "rng_state*", "scheduler.pt"))
    print(f"Adapter {src} -> {dest}")


# ------------------------------- SFT -------------------------------
def sft_aligntune(a, data, run_dir):
    from aligntune.core.backend_factory import create_sft_trainer
    tr = create_sft_trainer(
        model_name=BASE_MODEL, dataset_name=str(data), backend="trl", task_type="sft",
        output_dir=str(run_dir), num_epochs=a.epochs, batch_size=SFT_BS, gradient_accumulation_steps=16 // SFT_BS,
        learning_rate=2e-4, max_seq_length=MAX_LEN, max_samples=a.max_samples,
        use_peft=True, lora_r=16, lora_alpha=32, lora_dropout=0.05, lora_target_modules=LORA_TARGETS,
        quantization=QUANT, **PREC, warmup_ratio=0.05,
        completion_only_loss=True, assistant_only_loss=False, packing=False, group_by_length=False,
        gradient_checkpointing=True, use_gradient_checkpointing=True, load_best_model_at_end=False,
        save_strategy="steps", save_steps=SAVE_EVERY, save_total_limit=1, logging_steps=5, report_to="none")
    tr.train()


def sft_trl(a, data, run_dir):
    from datasets import load_dataset
    from trl import SFTConfig, SFTTrainer
    model, tok = load_train_model(BASE_MODEL)
    ds = load_dataset("json", data_files=str(data), split="train")
    if a.max_samples:
        ds = ds.select(range(min(a.max_samples, len(ds))))
    cfg = mk(SFTConfig, output_dir=str(run_dir), num_train_epochs=a.epochs, per_device_train_batch_size=SFT_BS,
             gradient_accumulation_steps=16 // SFT_BS, learning_rate=2e-4, max_length=MAX_LEN,
             fp16=not BF16, bf16=BF16,
             logging_steps=5, save_strategy="steps", save_steps=SAVE_EVERY, save_total_limit=1,
             warmup_ratio=0.05, lr_scheduler_type="cosine",
             gradient_checkpointing=True, completion_only_loss=True, optim="adamw_torch" if BIG else "paged_adamw_8bit",
             report_to="none")
    tr = SFTTrainer(model=model, args=cfg, train_dataset=ds, processing_class=tok, peft_config=lora())
    tr.train()
    tr.save_model(str(run_dir))


# ------------------------------- DPO -------------------------------
# Newer TRL dropped rpo_alpha/max_prompt_length; the NLL-on-chosen term is loss_type ["sigmoid","sft"].
# mk() drops whichever spelling the installed TRL doesn't know. Reference log-probs are computed once up
# front (precompute_ref_log_probs) so training never holds the policy and reference passes together.
DPO_KW = dict(beta=0.5, rpo_alpha=0.2, max_length=MAX_LEN, max_prompt_length=MAX_PROMPT,
              max_completion_length=MAX_COMPLETION, truncation_mode="keep_start")


def dpo_aligntune(a, model_path, data, run_dir):
    from aligntune.core.backend_factory import create_rl_trainer
    tr = create_rl_trainer(
        model_name=str(model_path), dataset_name=str(data), algorithm="dpo", backend=a.dpo_backend,
        output_dir=str(run_dir), num_epochs=1, batch_size=DPO_BS, gradient_accumulation_steps=16 // DPO_BS,
        learning_rate=1e-5, max_seq_length=MAX_LEN, max_samples=a.max_samples,
        use_peft=True, lora_r=16, lora_alpha=32, lora_dropout=0.05, lora_target_modules=LORA_TARGETS,
        quantization=QUANT, **PREC, warmup_ratio=0.1,
        gradient_checkpointing=True, use_gradient_checkpointing=True, report_to="none", logging_steps=5,
        save_strategy="steps", save_steps=SAVE_EVERY, save_total_limit=1,
        loss_type=["sigmoid", "sft"], loss_weights=[1.0, 0.2], **DPO_KW)
    tr.train()


def dpo_loss_kw(cls):
    """RPO-style NLL term: TRL >= 1.0 spells it as a weighted 'sft' loss, older TRL as rpo_alpha (in DPO_KW)."""
    names = {f.name for f in dataclasses.fields(cls)}
    return {"loss_type": ["sigmoid", "sft"], "loss_weights": [1.0, 0.2]} if "loss_weights" in names else {}


def dpo_trl(a, model_path, data, run_dir):
    from datasets import load_dataset
    from trl import DPOConfig, DPOTrainer
    model, tok = load_train_model(model_path)
    ds = load_dataset("json", data_files=str(data), split="train")
    if a.max_samples:
        ds = ds.select(range(min(a.max_samples, len(ds))))
    cfg = mk(DPOConfig, output_dir=str(run_dir), num_train_epochs=1, per_device_train_batch_size=DPO_BS,
             gradient_accumulation_steps=16 // DPO_BS, learning_rate=1e-5, fp16=not BF16, bf16=BF16,
             logging_steps=5,
             save_strategy="steps", save_steps=SAVE_EVERY, save_total_limit=1, warmup_ratio=0.1,
             lr_scheduler_type="cosine", gradient_checkpointing=True,
             optim="adamw_torch" if BIG else "paged_adamw_8bit", report_to="none",
             **dpo_loss_kw(DPOConfig), **DPO_KW)
    tr = DPOTrainer(model=model, ref_model=None, args=cfg, train_dataset=ds, processing_class=tok,
                    peft_config=lora())
    tr.train()
    tr.save_model(str(run_dir))


def run(primary, fallback, engine, *args):
    if engine == "aligntune":
        try:
            return primary(*args)
        except Exception:
            traceback.print_exc()
            print("\n!! AlignTune path failed - falling back to plain TRL (same settings).\n")
    import gc
    import torch
    gc.collect()
    torch.cuda.empty_cache()
    fallback(*args)


def merge():
    from peft import PeftModel
    from transformers import AutoTokenizer
    from common.model import load_causal_lm
    src = OUT / "adapter_sft"
    base = load_causal_lm(BASE_MODEL, device_map={"": 0})
    model = PeftModel.from_pretrained(base, str(src)).merge_and_unload()
    MERGED_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(MERGED_DIR), max_shard_size="2GB")
    AutoTokenizer.from_pretrained(BASE_MODEL).save_pretrained(str(MERGED_DIR))
    print(f"Merged SFT model -> {MERGED_DIR}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--stage", choices=["sft", "merge", "dpo"], required=True)
    ap.add_argument("--engine", choices=["aligntune", "trl"], default="aligntune")
    ap.add_argument("--dpo-backend", choices=["unsloth", "trl"], default="trl")
    ap.add_argument("--epochs", type=int, default=2)
    ap.add_argument("--max-samples", type=int, default=None)
    ap.add_argument("--smoke", action="store_true", help="tiny run into outputs/smoke/ to test the setup")
    a = ap.parse_args()
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
    root = OUT / "smoke" if a.smoke else OUT
    if a.smoke:
        a.max_samples, a.epochs = a.max_samples or 24, 1
        for d in ("run_sft", "run_dpo"):
            shutil.rmtree(root / d, ignore_errors=True)
    print("GPU:", describe())
    if not BF16:
        fp32_trainable()

    if a.stage == "sft":
        run_dir = root / "run_sft"
        enable_resume(run_dir)
        run(sft_aligntune, sft_trl, a.engine, a, OUT / "sft" / "train.jsonl", run_dir)
        collect_adapter(run_dir, root / "adapter_sft")
    elif a.stage == "merge":
        merge()
    else:
        model_path = BASE_MODEL if a.smoke else MERGED_DIR
        if not a.smoke and not (MERGED_DIR / "config.json").exists():
            sys.exit(f"{MERGED_DIR} missing - run --stage merge first")
        run_dir = root / "run_dpo"
        enable_resume(run_dir)
        data = short_pairs(OUT / "pref" / "train.jsonl", model_path, root / "pref_dpo.jsonl", a.max_samples)
        run(dpo_aligntune, dpo_trl, a.engine, a, model_path, data, run_dir)
        collect_adapter(run_dir, root / "adapter_dpo")
    import torch
    if torch.cuda.is_available():
        print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/06_generate.py
"""Step 6 (v2) - run 4 systems on the test set. All see the IDENTICAL retrieved passages per question.

  base_rag     : base model + simple prompt
  base_strict  : base model + the quote-then-answer grounding prompt (strong baseline, same prompt as ours)
  tuned_sft    : base + SFT adapter            (ablation: what SFT alone gives)
  tuned_dpo    : merged SFT + DPO adapter      (final model)

Resumable: a system whose preds file already exists is skipped (use --force to redo).
TEST_CSV env var lets you score on another test set (e.g. Sneha's) for a like-for-like comparison.
"""
import argparse
import csv
import json
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from common.model import Generator  # noqa: E402
from common.prompts import SYSTEM_QUOTE, SYSTEM_SIMPLE, build_messages, final_answer  # noqa: E402
from common.retrieval import Index, is_neighbor, load_chunks  # noqa: E402
from config import BASE_MODEL, OUT, TOP_K  # noqa: E402

MERGED_DIR = Path(os.getenv("MERGED_DIR", "/content/merged_sft"))
# system -> (prompt, model group, use adapter)
SYSTEMS = {"base_rag": (SYSTEM_SIMPLE, "base", False),
           "base_strict": (SYSTEM_QUOTE, "base", False),
           "tuned_sft": (SYSTEM_QUOTE, "base", True),
           "tuned_dpo": (SYSTEM_QUOTE, "merged", True)}


def load_test():
    path = Path(os.getenv("TEST_CSV", OUT / "test_candidates.csv"))
    with open(path, encoding="utf-8-sig") as f:
        rows = [r for r in csv.DictReader(f) if str(r.get("keep", "1")).strip() not in ("0", "")]
    print(f"Test set {path}: {sum(r['type'] == 'answerable' for r in rows)} answerable, "
          f"{sum(r['type'] == 'unanswerable' for r in rows)} unanswerable")
    return rows


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--systems", default=",".join(SYSTEMS))
    ap.add_argument("--limit", type=int, default=None)
    ap.add_argument("--force", action="store_true")
    args = ap.parse_args()

    rows = load_test()[:args.limit]
    chunks = load_chunks()
    by_id = {c["chunk_id"]: c for c in chunks}
    index = Index(chunks)
    for r in rows:
        r["ctx"] = index.search(r["question"], k=TOP_K)
        gold = by_id.get(r["gold_chunk_id"])
        r["gold_pos"] = [i + 1 for i, c in enumerate(r["ctx"])
                         if gold and (c["chunk_id"] == gold["chunk_id"] or is_neighbor(c, gold))]
        r["retrieval_hit"] = bool(r["gold_pos"])
    ans = [r for r in rows if r["type"] == "answerable"]
    print(f"Retrieval recall@{TOP_K}: {sum(r['retrieval_hit'] for r in ans) / max(len(ans), 1):.1%}")

    pred_dir = Path(os.getenv("PREDS_DIR", OUT / "preds"))
    pred_dir.mkdir(parents=True, exist_ok=True)
    todo = [s for s in args.systems.split(",") if args.force or not (pred_dir / f"{s}.jsonl").exists()]
    print("to generate:", todo or "nothing (all done)")
    for group in ("base", "merged"):
        names = [s for s in todo if SYSTEMS[s][1] == group]
        if not names:
            continue
        if group == "base":
            adapter = OUT / "adapter_sft" if any(SYSTEMS[s][2] for s in names) else None
            gen = Generator(BASE_MODEL, adapter)
        else:
            gen = Generator(MERGED_DIR, OUT / "adapter_dpo")
        for name in names:
            system, _, tuned = SYSTEMS[name]
            print(f"\n== {name}")
            outs = gen.generate([build_messages(r["question"], r["ctx"], system=system) for r in rows],
                                tuned=tuned)
            with open(pred_dir / f"{name}.jsonl", "w", encoding="utf-8") as f:
                for r, o in zip(rows, outs):
                    f.write(json.dumps({
                        "id": r["id"], "type": r["type"], "kind": r["kind"], "question": r["question"],
                        "gold_answer": r["gold_answer"], "retrieval_hit": r["retrieval_hit"],
                        "gold_pos": r["gold_pos"], "ctx_ids": [c["chunk_id"] for c in r["ctx"]],
                        "ctx": [f"[S{i + 1}] {c['text']}" for i, c in enumerate(r["ctx"])],
                        "output": o, "answer": final_answer(o)}, ensure_ascii=False) + "\n")
            print(outs[0][:400])
        gen.free()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/07_evaluate.py
"""Step 7 (v2) - score all systems with three independent checks + paired statistics.

  1. LLM judge from a DIFFERENT family than the Qwen student (default Gemma-3-12B on Ollama):
     refused? correct vs gold? number of claims unsupported by the sources?
  2. HHEM-2.1-open (Vectara, 0.1B NLI-style checker, no LLM): is the answer entailed by the sources?
  3. Deterministic checks (no judge): exact refusal string, citation correctness vs the gold passage,
     quotes copied verbatim from the cited source.
  + AuditKIT metrics and ak.compare(base_strict, tuned_dpo).

Headline metrics (Trust-Score style, Song et al. ICLR 2025): grounded-refusal F1 (balances refusing
when evidence is missing against answering when it is present), answer rate, over-refusal, accuracy,
hallucination among answered, citation precision/recall. Paired McNemar test + paired bootstrap CIs.

    python scripts/07_evaluate.py [--extra /path/to/other/preds:label ...] [--no-auditkit]
Writes outputs/results/{metrics.json, report.md, chart.png, judged_<system>.jsonl}
"""
import argparse
import os
import json
import math
import random
import re
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from common.llm import chat, parse_json, pmap  # noqa: E402
from common.prompts import REFUSAL, cited_ids, final_answer, is_refusal, quote_is_verbatim, quotes  # noqa: E402
from config import AUDIT_JUDGE, JUDGE_MODEL, OUT, TOP_K  # noqa: E402

LABELS = {"base_rag": "Base + simple prompt", "base_strict": "Base + grounding prompt",
          "tuned_sft": "Ours: SFT", "tuned_dpo": "Ours: SFT + DPO"}
RES = Path(os.getenv("RESULTS_DIR", OUT / "results"))
PREDS = Path(os.getenv("PREDS_DIR", OUT / "preds"))

JUDGE_PROMPT = """You grade a question-answering assistant that must answer ONLY from the given sources.

SOURCES:
{sources}

QUESTION: {question}
REFERENCE ANSWER: {gold}
ASSISTANT ANSWER: {output}

Decide:
- refused: true if the assistant says the information is not available / declines, WITHOUT giving a
  substantive answer. (An answer that corrects a false premise using the sources is NOT a refusal.)
- correct: compare with the reference. "yes" = same key facts, "partial" = some key facts or minor
  error, "no" = wrong or missing, "na" = no reference or the assistant refused.
- unsupported_claims: number of factual claims in the ASSISTANT ANSWER that the SOURCES do not
  support (outside knowledge counts as unsupported even if true). 0 for a refusal.
Return JSON only: {{"refused": true|false, "correct": "yes"|"partial"|"no"|"na", "unsupported_claims": <int>, "note": "<10 words>"}}"""


def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f]


def judge(p):
    gold = p["gold_answer"] if p["type"] == "answerable" else \
        "None - the sources are not expected to contain the answer; the ideal response is a refusal."
    j = parse_json(chat(JUDGE_MODEL, [{"role": "user", "content": JUDGE_PROMPT.format(
        sources="\n\n".join(p["ctx"]), question=p["question"], gold=gold, output=p["answer"])}],
        max_tokens=600))
    if not j:
        return None
    try:
        n = int(j.get("unsupported_claims") or 0)
    except (TypeError, ValueError):
        n = 0
    return {"refused": bool(j.get("refused")), "correct": str(j.get("correct", "na")).lower(),
            "unsupported_claims": n, "note": j.get("note", "")}


# ---------------- HHEM ----------------
_hhem = None


def hhem_scores(pairs):
    """P(consistent) for (premise, hypothesis) pairs; None if HHEM can't load."""
    global _hhem
    try:
        if _hhem is None:
            from transformers import AutoModelForSequenceClassification
            _hhem = AutoModelForSequenceClassification.from_pretrained(
                "vectara/hallucination_evaluation_model", trust_remote_code=True)
            try:
                _hhem = _hhem.to("cuda")
            except Exception:
                pass
            _hhem.eval()
        out = []
        for i in range(0, len(pairs), 8):
            s = _hhem.predict(pairs[i:i + 8])
            out += [float(x) for x in (s.tolist() if hasattr(s, "tolist") else s)]
        return out
    except Exception as e:
        print(f"HHEM unavailable ({str(e)[:200]}); skipping that check.")
        return None


# ---------------- stats ----------------
def rate(xs):
    xs = [x for x in xs if x is not None]
    return sum(xs) / len(xs) if xs else float("nan")


def boot_ci(xs, n=2000, seed=0):
    xs = [x for x in xs if x is not None]
    if not xs:
        return [float("nan")] * 2
    rng = random.Random(seed)
    b = sorted(sum(rng.choice(xs) for _ in xs) / len(xs) for _ in range(n))
    return [b[int(0.025 * n)], b[int(0.975 * n)]]


def f1(p, r):
    return 2 * p * r / (p + r) if p + r else 0.0


def mcnemar(a, b):
    """Exact two-sided McNemar p-value on paired booleans (a=baseline good, b=ours good)."""
    n01 = sum((not x) and y for x, y in zip(a, b))
    n10 = sum(x and (not y) for x, y in zip(a, b))
    n = n01 + n10
    if n == 0:
        return 1.0, n01, n10
    k = min(n01, n10)
    p = min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n)
    return p, n01, n10


def good(r):
    """Per-item 'did the right thing'."""
    if r["type"] == "unanswerable":
        return r["refused"]
    if r["refused"]:
        return not r.get("retrieval_hit", True)          # refusing is fine only if evidence wasn't retrieved
    return r["j"]["correct"] in ("yes", "partial") and r["j"]["unsupported_claims"] == 0


def enrich(p, v):
    r = {**p, "j": v}
    r.setdefault("answer", final_answer(p["output"]))
    # refusal = exact-string / heuristic OR judge says so
    r["refused"] = is_refusal(p["output"]) or v["refused"]
    r["exact_refusal"] = r["answer"].strip() == REFUSAL
    r["unsupported"] = (not r["refused"]) and v["unsupported_claims"] > 0
    return r


def metrics(rows):
    A = [r for r in rows if r["type"] == "answerable"]
    U = [r for r in rows if r["type"] == "unanswerable"]
    # Trust-Score style: an item "should be answered" iff it's answerable AND the evidence was retrieved
    should = [r for r in rows if r["type"] == "answerable" and r.get("retrieval_hit", True)]
    shouldnt = [r for r in rows if r not in should]
    answered = [r for r in rows if not r["refused"]]
    refused = [r for r in rows if r["refused"]]
    p_ans = rate([r in should for r in answered]) if answered else 0.0
    r_ans = rate([not r["refused"] for r in should]) if should else 0.0
    p_ref = rate([r in shouldnt for r in refused]) if refused else 0.0
    r_ref = rate([r["refused"] for r in shouldnt]) if shouldnt else 0.0
    cit = [r for r in answered if r in should and r.get("gold_pos")]
    cprec = [len(set(cited_ids(r["answer"])) & set(r["gold_pos"])) / len(cited_ids(r["answer"]))
             if cited_ids(r["answer"]) else 0.0 for r in cit]
    crec = [bool(set(cited_ids(r["answer"])) & set(r["gold_pos"])) for r in cit]
    qrows = [(r, q) for r in answered for q in quotes(r["output"])]
    qv = [quote_is_verbatim(q, r["ctx"][sid - 1] if 0 < sid <= len(r["ctx"]) else "") for r, (sid, q) in qrows]
    hall_all = [r["unsupported"] for r in rows]
    m = {
        "n": len(rows), "n_answerable": len(A), "n_unanswerable": len(U),
        "grounded_refusal_f1": (f1(p_ans, r_ans) + f1(p_ref, r_ref)) / 2,
        "answer_rate": len(answered) / len(rows),
        "correct_refusal_rate": rate([r["refused"] for r in U]),
        "correct_refusal_ci95": boot_ci([r["refused"] for r in U]),
        "exact_refusal_string": rate([r["exact_refusal"] for r in refused]),
        "over_refusal_rate": rate([r["refused"] for r in A]),
        "over_refusal_when_evidence_retrieved": rate([r["refused"] for r in should]),
        "accuracy": rate([r["j"]["correct"] == "yes" for r in A]),
        "accuracy_incl_partial": rate([r["j"]["correct"] in ("yes", "partial") for r in A]),
        "hallucination_rate": rate(hall_all), "hallucination_rate_ci95": boot_ci(hall_all),
        "hallucination_among_answered": rate([r["unsupported"] for r in answered]),
        "made_up_answer_on_unanswerable": rate([not r["refused"] for r in U]),
        "hhem_unsupported_among_answered": rate([r["hhem"] < 0.5 for r in answered if r.get("hhem") is not None]),
        "citation_rate": rate([bool(cited_ids(r["answer"])) for r in answered]),
        "citation_precision": rate(cprec), "citation_recall": rate(crec),
        "quote_verbatim_rate": rate(qv),
        "good_rate": rate([good(r) for r in rows]),
        "retrieval_recall_at_k": rate([r.get("retrieval_hit") for r in A]),
    }
    return m


def run_auditkit(rows_by_sys, use_llm_judge):
    import auditkit as ak
    scorers = ["lexical_groundedness", "context_coverage", "f1_score"]
    if use_llm_judge:
        from auditkit.metrics.rag_judge import Hallucination
        scorers = [Hallucination(judge_model=AUDIT_JUDGE, max_context_chars=2500)] + scorers
    runs, heads = {}, {}
    for name, rows in rows_by_sys.items():
        samples = [ak.Sample(input=r["question"], target=r["gold_answer"], id=r["id"],
                             retrieval_context=r["ctx"], actual_output=r["answer"],
                             metadata={"type": r["type"]}) for r in rows]
        res = ak.evaluate(samples, "precomputed", scorers, experiment_name=f"grounded-v2-{name}")
        runs[name], heads[name] = res, dict(res.headline)
        print(f"AuditKIT {name}: {res.headline}")
    if "base_strict" in runs and "tuned_dpo" in runs:
        try:
            cmp = ak.compare(runs["base_strict"], runs["tuned_dpo"])
            txt = cmp.summary() if hasattr(cmp, "summary") else str(cmp)
            (RES / "auditkit_compare.txt").write_text(str(txt), encoding="utf-8")
            print(txt)
        except Exception as e:
            print(f"AuditKIT compare failed: {e}")
    return heads


def chart(all_m):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    keys = [("grounded_refusal_f1", "Grounded-refusal F1 ↑"),
            ("hallucination_among_answered", "Hallucination (answered) ↓"),
            ("correct_refusal_rate", "Correct refusal ↑"),
            ("over_refusal_when_evidence_retrieved", "Over-refusal ↓"),
            ("accuracy_incl_partial", "Accuracy ↑")]
    systems = list(all_m)
    colors = ["#9aa5b1", "#5b7083", "#7fb3d5", "#1f9d55", "#c0392b", "#8e44ad"]
    fig, axes = plt.subplots(1, len(keys), figsize=(18, 4))
    for ax, (k, title) in zip(axes, keys):
        vals = [(all_m[s][k] if all_m[s][k] == all_m[s][k] else 0) * 100 for s in systems]
        bars = ax.bar(range(len(systems)), vals, color=colors[:len(systems)])
        ax.set_title(title, fontsize=10)
        ax.set_xticks(range(len(systems)))
        ax.set_xticklabels([LABELS.get(s, s).replace(": ", ":\n") for s in systems], fontsize=7)
        ax.set_ylim(0, 105)
        ax.spines[["top", "right"]].set_visible(False)
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v + 1.5, f"{v:.0f}", ha="center", fontsize=8)
    plt.tight_layout()
    plt.savefig(RES / "chart.png", dpi=160)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--extra", nargs="*", default=[], help="other preds files as path.jsonl:label")
    ap.add_argument("--no-auditkit", action="store_true")
    ap.add_argument("--no-auditkit-judge", action="store_true")
    args = ap.parse_args()
    RES.mkdir(parents=True, exist_ok=True)

    preds = {s: read_jsonl(PREDS / f"{s}.jsonl") for s in LABELS if (PREDS / f"{s}.jsonl").exists()}
    for e in args.extra:
        path, label = e.rsplit(":", 1)
        preds[label] = read_jsonl(path)
        LABELS[label] = label
    rows_by_sys, all_m = {}, {}
    for name, ps in preds.items():
        for p in ps:
            p.setdefault("answer", final_answer(p["output"]))
        verdicts = pmap(judge, ps, desc=f"judge {name}")
        rows = [enrich(p, v) for p, v in zip(ps, verdicts) if v is not None]
        if len(rows) < len(ps):
            print(f"  {len(ps) - len(rows)} judge failures skipped")
        ans = [r for r in rows if not r["refused"]]
        hs = hhem_scores([("\n\n".join(r["ctx"])[:8000], re.sub(r"\[S\d+\]", "", r["answer"]).strip())
                          for r in ans]) if ans else []
        for r, h in zip(ans, hs or []):
            r["hhem"] = h
        with open(RES / f"judged_{name}.jsonl", "w", encoding="utf-8") as f:
            for r in rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        rows_by_sys[name], all_m[name] = rows, metrics(rows)

    # paired comparisons vs the strong baseline
    tests = {}
    if "base_strict" in rows_by_sys:
        base = {r["id"]: r for r in rows_by_sys["base_strict"]}
        for name, rows in rows_by_sys.items():
            if name == "base_strict":
                continue
            ids = [r["id"] for r in rows if r["id"] in base]
            mine = {r["id"]: r for r in rows}
            p, n01, n10 = mcnemar([good(base[i]) for i in ids], [good(mine[i]) for i in ids])
            diffs = [int(good(mine[i])) - int(good(base[i])) for i in ids]
            tests[name] = {"good_rate_gain": sum(diffs) / max(len(diffs), 1), "gain_ci95": boot_ci(diffs),
                           "mcnemar_p": p, "fixed": n01, "broke": n10, "n_paired": len(ids)}

    if not args.no_auditkit:
        try:
            for name, h in run_auditkit(rows_by_sys, not args.no_auditkit_judge).items():
                all_m[name]["auditkit"] = h
        except Exception as e:
            print(f"AuditKIT step failed ({e}); other metrics are still saved.")

    json.dump({"metrics": all_m, "paired_vs_base_strict": tests}, open(RES / "metrics.json", "w"), indent=2)
    pct = lambda x: "-" if x is None or x != x else f"{x * 100:.1f}"  # noqa: E731
    cols = list(all_m)
    lines = ["| Metric | " + " | ".join(LABELS.get(s, s) for s in cols) + " |", "|---|" + "---|" * len(cols)]
    for k, label in [("grounded_refusal_f1", "Grounded-refusal F1 (Trust-Score style) ↑"),
                     ("good_rate", "Did the right thing (per item) ↑"),
                     ("hallucination_among_answered", "Hallucination among answered (judge) ↓"),
                     ("hhem_unsupported_among_answered", "Unsupported among answered (HHEM) ↓"),
                     ("hallucination_rate", "Responses with an unsupported claim ↓"),
                     ("correct_refusal_rate", "Correct refusal on unanswerable ↑"),
                     ("made_up_answer_on_unanswerable", "Answered an unanswerable question ↓"),
                     ("over_refusal_when_evidence_retrieved", "Over-refusal when evidence retrieved ↓"),
                     ("over_refusal_rate", "Over-refusal on all answerable ↓"),
                     ("accuracy", "Accuracy on answerable (strict) ↑"),
                     ("accuracy_incl_partial", "Accuracy on answerable (incl. partial) ↑"),
                     ("citation_precision", "Citation precision (cites the gold passage) ↑"),
                     ("citation_recall", "Citation recall ↑"),
                     ("quote_verbatim_rate", "Quotes copied verbatim ↑"),
                     ("exact_refusal_string", "Refusals using the exact sentence ↑"),
                     ("answer_rate", "Answer rate")]:
        lines.append(f"| {label} | " + " | ".join(pct(all_m[s].get(k)) for s in cols) + " |")
    m0 = next(iter(all_m.values()))
    lines.append(f"\nAll numbers are %. Test set: {m0['n_answerable']} answerable + {m0['n_unanswerable']} "
                 f"unanswerable. Retrieval recall@{TOP_K}: {pct(m0['retrieval_recall_at_k'])}%. "
                 f"Judge: {JUDGE_MODEL} (different family from the Qwen model being trained) + HHEM-2.1-open.")
    lines.append("\n**Paired vs Base + grounding prompt** (same questions, same passages):\n")
    for name, t in tests.items():
        lo, hi = t["gain_ci95"]
        lines.append(f"- {LABELS.get(name, name)}: right-thing rate {t['good_rate_gain'] * 100:+.1f} pts "
                     f"(95% CI {lo * 100:+.1f} to {hi * 100:+.1f}); fixed {t['fixed']}, broke {t['broke']}; "
                     f"McNemar p = {t['mcnemar_p']:.3g}")
    report = "\n".join(lines)
    (RES / "report.md").write_text(report, encoding="utf-8")
    print("\n" + report)
    chart(all_m)
    print(f"\nSaved {RES}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile app.py
"""Gradio demo: base model vs fine-tuned model, side by side, same retrieved sources.
Cited sources are highlighted. Run on Kaggle/Colab GPU:  python app.py   (prints a public link)
"""
import csv
import html
import random
import re

import gradio as gr

import os
from pathlib import Path

from common.model import Generator
from common.prompts import REFUSAL, SYSTEM_QUOTE, build_messages, cited_ids, final_answer
from common.retrieval import Index, load_chunks
from config import BASE_MODEL, OUT, TOP_K

chunks = load_chunks()
index = Index(chunks)
# v2: base model and final (merged SFT + DPO adapter) model, both 4-bit so two fit on one T4
base_gen = Generator(BASE_MODEL, None, load_in_4bit=True)
tuned_gen = Generator(Path(os.getenv("MERGED_DIR", "/content/merged_sft")), OUT / "adapter_dpo", load_in_4bit=True)

examples = []
try:
    with open(OUT / "test_candidates.csv", encoding="utf-8-sig") as f:
        rows = [r for r in csv.DictReader(f) if r.get("keep", "1") != "0"]
    random.Random(1).shuffle(rows)
    examples = [r["question"] for r in rows if r["type"] == "answerable"][:5] + \
               [r["question"] for r in rows if r["type"] == "unanswerable"][:5]
except FileNotFoundError:
    pass


def render_answer(text):
    if final_answer(text).strip().startswith(REFUSAL):
        return f"<div class='ans refusal'>🛑 {html.escape(text)}</div>"
    body = re.sub(r"\[S(\d+)\]", r"<span class='cite'>[S\1]</span>", html.escape(text)).replace("\n", "<br>")
    return f"<div class='ans'>{body}</div>"


def render_sources(ctx, cited):
    out = []
    for i, c in enumerate(ctx, 1):
        cls = "src cited" if i in cited else "src"
        out.append(f"<div class='{cls}'><b>[S{i}]</b> <i>{html.escape(c['doc'])}, p.{c['page']}</i>"
                   f"{' ✅ cited' if i in cited else ''}<br>{html.escape(c['text'])}</div>")
    return "".join(out)


def ask(question):
    if not question.strip():
        return "", "", ""
    ctx = index.search(question, k=TOP_K)
    msgs = build_messages(question, ctx, system=SYSTEM_QUOTE)
    base = base_gen.generate([msgs], tuned=False)[0]
    tuned = tuned_gen.generate([msgs], tuned=True)[0]
    return render_answer(base), render_answer(tuned), render_sources(ctx, set(cited_ids(tuned)))


CSS = """
.ans{padding:12px;border-radius:8px;background:#f4f6f8;color:#111;font-size:15px;line-height:1.5}
.refusal{background:#fff4e5;border-left:4px solid #f59e0b}
.cite{background:#d1fae5;color:#065f46;padding:0 4px;border-radius:4px;font-weight:600}
.src{padding:10px;margin:6px 0;border:1px solid #e5e7eb;border-radius:8px;font-size:13px;color:#222;background:#fff}
.cited{border:2px solid #10b981;background:#ecfdf5}
"""

with gr.Blocks(css=CSS, title="Grounded Carbon-Market QA") as demo:
    gr.Markdown("# Grounded Carbon-Market QA\nThe same retrieved passages go to both models. "
                "The fine-tuned model answers only from them, cites them, or says **Not found**.")
    q = gr.Textbox(label="Question", placeholder="e.g. How long is the minimum crediting period under ...?")
    btn = gr.Button("Ask", variant="primary")
    with gr.Row():
        with gr.Column():
            gr.Markdown("### Base model (same grounding prompt)")
            a_base = gr.HTML()
        with gr.Column():
            gr.Markdown("### Fine-tuned model (ours: SFT + DPO)")
            a_tuned = gr.HTML()
    gr.Markdown("### Retrieved sources (green = cited by the fine-tuned model)")
    srcs = gr.HTML()
    if examples:
        gr.Examples(examples, inputs=q)
    btn.click(ask, q, [a_base, a_tuned, srcs])
    q.submit(ask, q, [a_base, a_tuned, srcs])

if __name__ == "__main__":
    demo.launch(share=True)

## 3. Install (AlignTune ships its own Unsloth, so it is not installed separately)

In [ ]:
!pip install -q aligntune rank_bm25 sentence-transformers litellm python-dotenv auditkit "curatorkit[all]" 2>&1 | tail -3
!pip install -q "trl==1.18.0" 2>&1 | tail -2   # scores only answer tokens -> DPO fits the T4 (AlignTune pins 1.7.1)
!python -c "import aligntune, trl, transformers, torch; print('ok', torch.cuda.get_device_name(0), 'trl', trl.__version__, 'transformers', transformers.__version__)"

## 4. Local LLM server (Ollama) for building the data

In [ ]:
!apt-get -qq install -y zstd pciutils > /dev/null; curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import requests
def start_ollama():
    p = subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
    for _ in range(60):
        try: requests.get('http://localhost:11434'); return p
        except Exception: time.sleep(1)
def stop_ollama():
    global ollama
    try: ollama.terminate(); ollama.wait(timeout=20)
    except Exception: pass
    subprocess.run(['pkill', '-f', 'ollama']); time.sleep(5)
ollama = start_ollama()
need_data = not os.path.exists(f'{O}/pref/train.jsonl')
if need_data:
    !ollama pull qwen2.5:14b 2>&1 | tail -1
    !python -c "from common.llm import chat; import os; print(chat(os.environ['GEN_MODEL'], [{'role':'user','content':'Say OK'}], max_tokens=10))"
else:
    print('Data already built - skipping model download')

## 5. Build the data (≈45–60 min the first time; skipped if already on Drive)
QA pairs + hallucination gate → verified unanswerable questions → v2 SFT set + DPO pairs + test set.

In [ ]:
if not os.path.exists(f'{W}/qa_raw.jsonl'):
    !python scripts/02_curate.py --engine fallback
if not os.path.exists(f'{W}/unanswerable.jsonl'):
    !python scripts/03_unanswerable.py
if not os.path.exists(f'{O}/pref/train.jsonl'):
    !python scripts/04_build_sets.py
print(open(f'{W}/sft_stats_v2.json').read())
print('example SFT target:\n', json.loads(open(f'{O}/sft/train.jsonl').readline())['completion'][0]['content'][:500])

## 6. Free the GPU for training

In [ ]:
stop_ollama()
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## 7. Smoke tests (~5 min): prove SFT and DPO run before the long jobs
Speed check: the SFT smoke's 2 steps should take **~2–4 min in total** (not 12). If you see *“AlignTune path failed - falling back to plain TRL”*, that's OK: the same settings run on TRL.

In [ ]:
if os.path.exists(f'{O}/smoke/adapter_dpo/adapter_config.json') or os.path.exists(f'{O}/adapter_sft/adapter_config.json'):
    print('smoke tests already passed - skipping')
else:
    !python scripts/05_train.py --stage sft --smoke
    !python scripts/05_train.py --stage dpo --smoke

## 8. SFT: quote-then-answer, loss on the answer only (≈1.5–2 h)
Saves a checkpoint to Drive every 10 steps; after a disconnect just run this cell again and it resumes.

In [ ]:
if not os.path.exists(f'{O}/adapter_sft/adapter_config.json'):
    !python scripts/05_train.py --stage sft --epochs 2
else:
    print('SFT adapter found on Drive - skipping')

## 9. Merge SFT into the base model (≈5 min; local disk, redone after a disconnect)

In [ ]:
if not os.path.exists('/content/merged_sft/config.json') and not os.path.exists(f'{O}/preds/tuned_dpo.jsonl'):
    !python scripts/05_train.py --stage merge
else:
    print('merged model present (or not needed) - skipping')

## 10. DPO on the preference pairs that fit in 2048 tokens (≈1–1.5 h)
Reference log-probs are computed first (a progress bar), then training. Checkpoints every 10 steps; rerun to resume.

In [ ]:
if not os.path.exists(f'{O}/adapter_dpo/adapter_config.json'):
    !python scripts/05_train.py --stage dpo
else:
    print('DPO adapter found on Drive - skipping')

## 11. Answer the test questions with 4 systems (≈30–45 min, resumable per system)

In [ ]:
need_merged = not os.path.exists(f'{O}/preds/tuned_dpo.jsonl')
if need_merged and not os.path.exists('/content/merged_sft/config.json'):
    !python scripts/05_train.py --stage merge
!python scripts/06_generate.py

## 12. Score: Gemma judge + HHEM + exact checks + AuditKIT (≈30–45 min)

In [ ]:
import gc, torch; gc.collect(); torch.cuda.empty_cache()
ollama = start_ollama()
!ollama pull gemma3:12b 2>&1 | tail -1
!python scripts/07_evaluate.py
from IPython.display import Image, Markdown, display
display(Markdown(open(f'{O}/results/report.md').read())); display(Image(f'{O}/results/chart.png'))

## 13. Save results (also kept on Drive)

In [ ]:
os.chdir(O)
!zip -qr /content/karan_v2_results.zip sft pref test_candidates.csv preds results adapter_sft adapter_dpo
os.chdir('/content/proj')
shutil.copy('/content/karan_v2_results.zip', f'{SAVE}/karan_v2_results.zip')
from google.colab import files; files.download('/content/karan_v2_results.zip')

## 14. (Optional) Compare with Sneha's run on the SAME test questions
Set `RUN_COMPARE = True`, run the cell, and upload Sneha's `results.zip` (from her notebook).
It answers **her** test questions with our models, then scores her fine-tuned model and ours with the **same judge**.

In [ ]:
RUN_COMPARE = False   # <- set True once you have Sneha's results.zip
if RUN_COMPARE:
  from google.colab import files
  up = files.upload()                          # choose Sneha's results.zip
  !rm -rf /content/sneha && mkdir -p /content/sneha && unzip -q -o "{list(up)[0]}" -d /content/sneha
  os.environ['TEST_CSV'] = '/content/sneha/outputs/test_candidates.csv'
  os.environ['PREDS_DIR'] = f'{O}/preds_sneha_test'
  os.environ['RESULTS_DIR'] = f'{O}/results_vs_sneha'
  stop_ollama()
  if not os.path.exists('/content/merged_sft/config.json'):
      !python scripts/05_train.py --stage merge
  !python scripts/06_generate.py --systems base_strict,tuned_dpo
  ollama = start_ollama()
  !python scripts/07_evaluate.py --no-auditkit --extra /content/sneha/outputs/preds/tuned_rag.jsonl:sneha_tuned
  display(Markdown(open(f'{O}/results_vs_sneha/report.md').read()))
  for k in ['TEST_CSV', 'PREDS_DIR', 'RESULTS_DIR']: os.environ.pop(k)

## 15. Demo with a public link (stop the cell to end it)

In [ ]:
stop_ollama()
if not os.path.exists('/content/merged_sft/config.json'):
    !python scripts/05_train.py --stage merge
!pip install -q gradio
!python app.py